# 07 — Federated Learning (Simulated FedAvg): Taxonomy Classification\n
\n
Simulate federated clients by partitioning the FishNet training set into shards (stratified by species).\n
\n
Train via FedAvg and compare to centralized training.\n
\n
Implementation: minimal pure-PyTorch FedAvg loop (device-flexible).\n
Source reference (general FL concepts): https://flower.ai/docs/

In [ ]:
from pathlib import Path
import sys

import numpy as np
import torch

ROOT = Path('..').resolve()
sys.path.append(str((ROOT / 'src').resolve()))

MANIFEST_DIR = (ROOT / 'data' / 'manifests').resolve()
OUT_DIR = (ROOT / 'outputs' / 'federated_fedavg').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device(
    'cuda'
    if torch.cuda.is_available()
    else 'mps'
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
    else 'cpu'
)
print('device:', device)
print('OUT_DIR:', OUT_DIR)


In [ ]:
from torch.utils.data import DataLoader, Subset

from fish_ai.data.jsonl import read_jsonl
from fish_ai.data.taxonomy_dataset import FishTaxonomyDataset
from fish_ai.models.taxonomy_classifier import TaxonomyClassifier, TaxonomyHeadSizes
from fish_ai.train.fedavg import (
    FedAvgConfig,
    average_state_dicts,
    select_clients,
    split_indices_stratified,
)
from fish_ai.train.taxonomy_train import TrainConfig, build_label_maps, evaluate, train_one_epoch
from fish_ai.utils.run_logging import write_csv, write_json

train_manifest = MANIFEST_DIR / 'fishnet_taxonomy_train.jsonl'
val_manifest = MANIFEST_DIR / 'fishnet_taxonomy_val.jsonl'
print(train_manifest.exists(), val_manifest.exists())


In [ ]:
# Build label maps from FULL training set
train_rows = []
for r in read_jsonl(train_manifest):
    tax = r['taxonomy']
    train_rows.append({'family': tax['family'], 'genus': tax['genus'], 'species': tax['species']})

maps = build_label_maps(train_rows)

sizes = TaxonomyHeadSizes(
    n_family=len(maps['family']),
    n_genus=len(maps['genus']),
    n_species=len(maps['species']),
)

sizes


In [ ]:
# Datasets
ds_train_full = FishTaxonomyDataset(train_manifest, image_size=224, augment=True)
ds_val = FishTaxonomyDataset(val_manifest, image_size=224, augment=False)

# Client partitions (stratified by species)
species_ids = [maps['species'][r['species']] for r in train_rows]

fed_cfg = FedAvgConfig(
    num_clients=10,
    num_rounds=10,
    clients_per_round=None,
    local_epochs=1,
    seed=42,
)
client_splits = split_indices_stratified(species_ids, num_clients=fed_cfg.num_clients, seed=fed_cfg.seed)

[(i, len(s)) for i, s in enumerate(client_splits)][:5], sum(len(s) for s in client_splits)


In [ ]:
# Global model (start from transfer learning weights)
global_model = TaxonomyClassifier(sizes, backbone='efficientnet_b0', pretrained=True).to(device)
cfg = TrainConfig(num_epochs=fed_cfg.local_epochs, batch_size=64, num_workers=2, lr=3e-4)

dl_val = DataLoader(ds_val, batch_size=64, shuffle=False, num_workers=2)

history = []
for rnd in range(1, fed_cfg.num_rounds + 1):
    selected = select_clients(
        fed_cfg.num_clients,
        fed_cfg.clients_per_round,
        round_idx=rnd,
        seed=fed_cfg.seed,
    )
    client_states = []
    client_weights = []

    for cid in selected:
        idxs = client_splits[cid]
        if len(idxs) == 0:
            continue

        client_ds = Subset(ds_train_full, idxs)
        client_dl = DataLoader(
            client_ds,
            batch_size=cfg.batch_size,
            shuffle=True,
            num_workers=cfg.num_workers,
        )

        # Clone global weights into a client model
        client_model = TaxonomyClassifier(sizes, backbone='efficientnet_b0', pretrained=False).to(device)
        client_model.load_state_dict(global_model.state_dict())
        opt = torch.optim.AdamW(client_model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

        # Local training
        for _ in range(fed_cfg.local_epochs):
            _ = train_one_epoch(client_model, client_dl, opt, device=device, maps=maps)

        client_states.append({k: v.detach().cpu() for k, v in client_model.state_dict().items()})
        client_weights.append(float(len(idxs)))

    # FedAvg aggregation
    avg_state = average_state_dicts(client_states, weights=client_weights)
    global_model.load_state_dict(avg_state, strict=True)
    global_model.to(device)

    # Centralized validation
    val_metrics = evaluate(global_model, dl_val, device=device, maps=maps)
    row = {
        'round': rnd,
        'num_selected_clients': len(selected),
        'val_species_acc_top1': val_metrics['species']['acc_top1'],
        'val_species_f1_macro': val_metrics['species']['f1_macro'],
        'val_genus_acc_top1': val_metrics['genus']['acc_top1'],
        'val_family_acc_top1': val_metrics['family']['acc_top1'],
    }
    history.append(row)
    print(row)

ckpt_path = OUT_DIR / 'fedavg_global_model.pt'
torch.save(
    {
        'model_state': global_model.state_dict(),
        'backbone': global_model.backbone_name,
        'maps': maps,
        'fed_cfg': fed_cfg.__dict__,
    },
    ckpt_path,
)

write_csv(OUT_DIR / 'history.csv', history)
write_json(OUT_DIR / 'val_metrics_last.json', val_metrics)
print('saved:', ckpt_path)
print('wrote:', OUT_DIR / 'history.csv')
